# Bài 5 — Tracking đa đối tượng với ByteTrack

**Mục tiêu:** Gán ID cố định cho từng xe qua các frame — nền tảng của đếm và đo tốc độ. **Nhìn thấy ID bám theo xe ngay trên cửa sổ.**

## 0. Chuẩn bị (asset video + display.py dùng chung)

In [2]:
!pip install -q supervision ultralytics "supervision[assets]"

In [3]:
from supervision.assets import download_assets, VideoAssets

download_assets(VideoAssets.VEHICLES)
print(VideoAssets.VEHICLES.value)  # "vehicles.mp4"

[2026-08-18 10:24:32] [INFO] supervision.assets.downloader - vehicles.mp4 asset download complete.
vehicles.mp4


In [4]:
%%writefile display.py
# display.py — hàm hiển thị dùng chung cho toàn giáo trình
import cv2

WINDOW_NAME = "Supervision - Live"
MAX_DISPLAY_WIDTH = 1280   # thu nhỏ frame cho vừa màn hình (chỉ để XEM, không ảnh hưởng xử lý)


def show_frame(frame, window_name: str = WINDOW_NAME, wait: int = 1) -> bool:
    """Hiện frame lên cửa sổ. Trả về False nếu người dùng bấm Q/ESC (muốn thoát).

    wait=1  -> dùng cho video (hiện liên tục, không chặn)
    wait=0  -> dùng cho ảnh tĩnh (dừng lại chờ bấm phím bất kỳ)
    """
    h, w = frame.shape[:2]
    if w > MAX_DISPLAY_WIDTH:                      # thu nhỏ để vừa màn hình
        scale = MAX_DISPLAY_WIDTH / w
        frame = cv2.resize(frame, (int(w * scale), int(h * scale)))

    cv2.imshow(window_name, frame)
    key = cv2.waitKey(wait) & 0xFF
    if key in (ord("q"), ord("Q"), 27):            # Q hoặc ESC -> thoát
        return False
    return True


def close_windows():
    cv2.destroyAllWindows()

Overwriting display.py


## 5.1. Vấn đề: detection không có "trí nhớ"

Ở Bài 4, mỗi frame model detect độc lập — chiếc xe ở frame 10 và frame 11 là "hai đối tượng khác nhau". Muốn **đếm** hay **đo tốc độ**, phải biết "đây vẫn là xe số 7". Đó là bài toán **Multi-Object Tracking (MOT)**.

## 5.2. Thêm ByteTrack — chỉ 3 dòng

In [5]:
import numpy as np
import supervision as sv
from ultralytics import YOLO
from display import show_frame, close_windows

SOURCE_VIDEO = "vehicles.mp4"
TARGET_VIDEO = "bai5_output.mp4"
VEHICLE_CLASSES = [2, 3, 5, 7]
STRIDE = 1          # tăng lên 2-3 nếu máy yếu / kernel hay crash (bỏ bớt frame)
MAX_FRAMES = None   # đặt số nguyên (vd 300) để test nhanh 1 đoạn ngắn; None = chạy hết video

model = YOLO("yolov8n.pt")
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO)

#  Tracker: khớp detection giữa các frame, gán tracker_id cố định
tracker = sv.ByteTrack(
    frame_rate=video_info.fps,             # QUAN TRỌNG: khớp fps video
    track_activation_threshold=0.25,       # conf tối thiểu để khởi tạo track
    lost_track_buffer=30,                  # giữ track "mất dấu" trong 30 frame
    minimum_matching_threshold=0.8,        # ngưỡng IoU để khớp track
)

# Màu + nhãn theo tracker_id (không phải theo class) -> mỗi xe 1 màu riêng
box_annotator = sv.BoxAnnotator(thickness=2, color_lookup=sv.ColorLookup.TRACK)
label_annotator = sv.LabelAnnotator(text_scale=0.5, color_lookup=sv.ColorLookup.TRACK)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10696\2218753177.py:16: FutureWarning: The `ByteTrack` was deprecated since v0.28.0. It will be removed in v0.31.0.
  tracker = sv.ByteTrack(


## 5.3. Vẽ quỹ đạo với TraceAnnotator + 5.4. Smoothing

In [6]:
trace_annotator = sv.TraceAnnotator(
    trace_length=int(video_info.fps * 2),   # lưu vệt 2 giây
    position=sv.Position.BOTTOM_CENTER,
)

# Bật/tắt smoother để so sánh: có smoother box mượt hơn, tắt đi thì box rung lật phật
USE_SMOOTHER = True
smoother = sv.DetectionsSmoother(length=5)


def process_frame(frame: np.ndarray) -> np.ndarray:
    results = model(frame, imgsz=640, verbose=False)[0]   # imgsz cố định giúp nhẹ + ổn định hơn
    detections = sv.Detections.from_ultralytics(results)
    detections = detections[np.isin(detections.class_id, VEHICLE_CLASSES)]

    # Dòng ma thuật: cập nhật tracker -> detections.tracker_id có giá trị
    detections = tracker.update_with_detections(detections)

    if USE_SMOOTHER:
        detections = smoother.update_with_detections(detections)

    labels = [f"#{tid} {name}" for tid, name
              in zip(detections.tracker_id, detections.data["class_name"])]

    annotated = frame.copy()
    annotated = trace_annotator.annotate(annotated, detections)
    annotated = box_annotator.annotate(annotated, detections)
    annotated = label_annotator.annotate(annotated, detections, labels=labels)
    return annotated

>  **Hiểu sâu ByteTrack:** ByteTrack khớp track bằng IoU qua 2 vòng — vòng 1 với detection confidence cao, vòng 2 tận dụng cả detection confidence thấp (thường là xe bị che khuất) thay vì vứt đi. Đó là lý do nó giữ ID tốt khi xe che nhau. Lưu ý mỗi khi xử lý video mới phải `tracker.reset()`.

## Chạy pipeline — giữ nguyên bộ khung Bài 4 (VideoSink + show_frame)

In [7]:
tracker.reset()   # luôn reset trước khi chạy lại từ đầu video

sink = sv.VideoSink(target_path=TARGET_VIDEO, video_info=video_info)
sink.__enter__()

try:
    for i, frame in enumerate(sv.get_video_frames_generator(SOURCE_VIDEO, stride=STRIDE)):
        if MAX_FRAMES is not None and i >= MAX_FRAMES:
            print(f"Da chay du {MAX_FRAMES} frame (test) - dung.")
            break

        annotated = process_frame(frame)
        sink.write_frame(annotated)

        if not show_frame(annotated):            # bấm Q -> dừng sớm
            print("Nguoi dung bam Q - dung som.")
            break
except Exception as e:
    print("Loi trong luc xu ly:", e)
finally:
    sink.__exit__(None, None, None)   # đảm bảo file luôn được đóng đúng cách dù có lỗi/crash
    close_windows()

print("Xong! Video da luu tai:", TARGET_VIDEO)

Loi trong luc xu ly: 'class_name'
Xong! Video da luu tai: bai5_output.mp4


##  Checkpoint Bài 5

Trên cửa sổ live: mỗi xe một màu riêng, nhãn `#ID`, có vệt quỹ đạo bám theo. Quan sát trực tiếp thấy ID không đổi khi xe đi qua chỗ che khuất ngắn.

**Thử nghiệm:** đổi `USE_SMOOTHER = False` rồi chạy lại, so sánh độ "giật" của box giữa 2 lần chạy.